# NB07 Blocks and the chain — Chapter 7

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- Standard library only (run and checked on Python 3.13). There is no install cell.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB07-1: toy blockchain — a hash chain without proof of work. Tampering shows, but re-threading is free (for study)
import hashlib, json

def H(s: str) -> str:
    return hashlib.sha256(s.encode()).hexdigest()

class Block:
    def __init__(self, index, prev_hash, data, timestamp):
        self.index, self.prev_hash, self.data, self.timestamp = index, prev_hash, data, timestamp
    def header(self) -> str:                      # header = the part that goes into the chain
        return json.dumps([self.index, self.prev_hash, H(self.data), self.timestamp])
    def hash(self) -> str:                        # block hash = hash of the header (the data enters only as H(data), in the place of the Merkle root)
        return H(self.header())

def build_chain(datas):
    chain = [Block(0, "0" * 64, datas[0], 1231006505)]            # genesis
    for i, d in enumerate(datas[1:], start=1):
        chain.append(Block(i, chain[-1].hash(), d, 1231006505 + 600 * i))
    return chain

def validate(chain):
    """Check that each block's prev_hash equals the actual hash of the previous block. Return the number of the first wrong block (None if there is none)."""
    for i in range(1, len(chain)):
        if chain[i].prev_hash != chain[i - 1].hash():
            return i
    return None

datas = ["genesis", "A→B 1", "B→C 2", "C→D 3", "D→E 4", "E→F 5"]
chain = build_chain(datas)
print("number of blocks:", len(chain), "| valid?", validate(chain) is None)
for b in chain:
    print(f"  #{b.index} prev={b.prev_hash[:10]}… hash={b.hash()[:10]}… data={b.data!r}")

print("\n--- change the transaction in block 3 from '3' to '300' ---")
chain[3].data = "C→D 300"
bad = validate(chain)
print("first broken link: block #", bad, "(block 3 was changed, yet block 4 is reported wrong: block 4's prev_hash is the old hash)")
for b in chain[3:]:
    print(f"  #{b.index} prev={b.prev_hash[:10]}… actual previous hash={chain[b.index-1].hash()[:10]}…  match={b.prev_hash == chain[b.index-1].hash()}")

print("\n--- the tamperer rewrites the prev_hash of every later block ---")
for i in range(4, len(chain)):
    chain[i].prev_hash = chain[i - 1].hash()
print("re-threaded chain valid?", validate(chain) is None, "← without proof of work, relinking the whole chain takes one hash per block")

number of blocks: 6 | valid? True
  #0 prev=0000000000… hash=475e9da5b5… data='genesis'
  #1 prev=475e9da5b5… hash=e9668fba8f… data='A→B 1'
  #2 prev=e9668fba8f… hash=dcdec3cf8e… data='B→C 2'
  #3 prev=dcdec3cf8e… hash=daa1496cf4… data='C→D 3'
  #4 prev=daa1496cf4… hash=0e85befdef… data='D→E 4'
  #5 prev=0e85befdef… hash=3c35f3caa0… data='E→F 5'

--- change the transaction in block 3 from '3' to '300' ---
first broken link: block # 4 (block 3 was changed, yet block 4 is reported wrong: block 4's prev_hash is the old hash)
  #3 prev=dcdec3cf8e… actual previous hash=dcdec3cf8e…  match=True
  #4 prev=daa1496cf4… actual previous hash=a1b2cd168c…  match=False
  #5 prev=0e85befdef… actual previous hash=0e85befdef…  match=True

--- the tamperer rewrites the prev_hash of every later block ---
re-threaded chain valid? True ← without proof of work, relinking the whole chain takes one hash per block


In [ ]:
# NB07-2: when both chains are "valid" — hashes alone cannot tell which one is the original
original = build_chain(datas)
forged = build_chain([d if i != 3 else "C→D 300" for i, d in enumerate(datas)])
print("original chain valid?", validate(original) is None, "| forged chain valid?", validate(forged) is None)
print("last block hash — original:", original[-1].hash()[:16], "… forged:", forged[-1].hash()[:16], "…")
print("The two chains differ, but each is perfectly consistent internally. Hashes cannot answer 'which one is real?'")
print("What came in to answer this question is the proof of work (cost) of Chapter 8 and the rule of the 'chain with the most accumulated work'.")

original chain valid? True | forged chain valid? True
last block hash — original: 3c35f3caa070a93e … forged: 63f530852e4eed45 …
The two chains differ, but each is perfectly consistent internally. Hashes cannot answer 'which one is real?'
What came in to answer this question is the proof of work (cost) of Chapter 8 and the rule of the 'chain with the most accumulated work'.
